In [283]:
import numpy as np
from pathlib import Path
from itertools import groupby
import pandas as pd
import matplotlib.pyplot as plt

# **Loading spike data after Spike Sorting and Zhen Su's pipeline curation**

## Constructing bins - 200 ms time bins (5Hz)

In [ ]:
cluster_info = pd.read_csv(session_groups[6][0].parent.joinpath("cluster_info.tsv"), sep ="\t")
spikeClusters = np.load(session_groups[6][0].parent.joinpath("spike_clusters.npy")).astype(int)
spikeSeconds = np.load(session_groups[6][0].parent.joinpath("spike_seconds_imec0_adj.npy")).astype(float)

good_idx = np.where(cluster_info.group == "good")
goodClusters = np.array(cluster_info.cluster_id.loc[good_idx])
nGood = len(good_idx[0])
nBins = len(bins) - 1

perProbeFR = np.zeros(shape = (nGood, nBins))
for cell, clu in enumerate(goodClusters):
    perCellSpikes = spikeSeconds[spikeClusters == clu]
    perCellFR, _ = np.histogram(perCellSpikes, bins=bins)
    
    perProbeFR[cell, nBins] = perCellFR

c = np.concatenate((perProbeFR, perProbeFR), axis = 0)

In [ ]:
dtFR = 0.2 # 200 ms
tmin = 0.0
tmax = 1205.0 # From catGT -maxsecs=1205.00; So always fixed for all sessions
bins = np.arange(tmin, tmax + dtFR, dtFR)
GAUSS_FILTER = True

def session_FR(sessionProbeDataPaths, dtFR, tmin, tmax, bins, GAUSS_FILTER):

    for probePath in sessionProbeDataPaths:

        cluster_info = pd.read(probePath.parent.joinpath("cluster_info.tsv"), sep ="\t")
        spikeClusters = np.load(probePath.parent.joinpath("spike_clusters.npy")).astype(int)
        spikeSeconds = np.load(probePath.parent.joinpath("spike_seconds_imec0_adj.npy")).astype(float)

        good_idx = np.where(cluster_info.group == "good")
        goodClusters = np.array(cluster_info.cluster_id.loc[good_idx])
        nGood = len(good_idx[0])
        nBins = len(bins)

        perProbeFR = np.zeros(shape = (nGood, nBins))
        for cell, clu in iter(goodClusters):
            perCellSpikes = spikeSeconds[spikeClusters == clu]
            perCellFR, _ = np.histogram(perCellSpikes, bins=bins)
            
            perProbeFR[cell, :] = perCellFR

    return 

In [312]:
clu_spikes = spikeSeconds[spikeClusters == clu]

In [ ]:
all_counts = []

for clu in goodClusters:
    perCellSpikeTimes = spikeSeconds[spikeClusters == clu]
    perCellFR, _ = np.histogram(perCellSpikeTimes, bins=bins)
    all_counts.append(counts)

all_counts = np.concatenate(all_counts, axis = 1)

plt.figure(figsize=(6,4))
plt.hist(all_counts, bins=np.arange(all_counts.max() + 2) - 0.5, edgecolor='black')
plt.xlabel('Spike count per bin')
plt.ylabel('Number of cell-bins')
plt.title('Distribution of spike counts across all good cells')
plt.show()

AxisError: axis 1 is out of bounds for array of dimension 1

In [141]:
print("Sanity check:", np.array_equal(np.array(cluster_info.cluster_id), np.unique(spikeClusters)))

Sanity check: True


In [ ]:
# Grouping sessions
cluster_info = []
spike_times = []

for it, session in enumerate(session_groups):
